# Part 1: Data Cleaning and Validation

**Goal:** Load the raw table `ohlc_hw`, find data quality problems, repair what can be repaired, and save a clean table `ohlc_hw_cleaned`.

**Rules I follow**
- The raw table `ohlc_hw` is never changed.
- Every repair gets a flag column, so the change can be audited.
- Rows that cannot be repaired go to a quarantine table with a reason.
- This notebook can run again from the top on a fresh copy of `data.db`.

In [33]:
import sqlite3
import numpy as np
import pandas as pd

DB_PATH = "data/data.db"
conn = sqlite3.connect(DB_PATH)

raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
raw.shape

(397711, 8)

### Columns and their meaning

| Column | Meaning |
|---|---|
| `ts` | Observation date (one trading day). Stored as text in `YYYY-MM-DD` form. |
| `ticker` | Stock symbol. This column is dirty (see Section 1.2). |
| `open` | Raw price at the start of the trading day. |
| `high` | Raw highest price of the day. |
| `low` | Raw lowest price of the day. |
| `close` | Raw price at the end of the day. |
| `adj_close` | Vendor adjusted close. The vendor corrects old prices for events such as splits and dividends. I use it for returns in Part 2. |
| `volume` | Number of shares traded that day. |

"Raw" means the price was not adjusted for splits and dividends.

In [34]:
raw.info()
raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


### What the statistics show

I write my own notes here after I read the output above. Things to check:
- Is any `min` value zero or negative for a price column?
- Is `volume` ever zero?
- Does `count` differ between columns? (A lower count means missing values.)

In [35]:
print("First date:", raw["ts"].min())
print("Last date: ", raw["ts"].max())
print("Number of ticker labels:", raw["ticker"].nunique())
print("First 5 labels:")
for label in sorted(raw["ticker"].unique())[:5]:
    print("  ", label)

First date: 1962-01-02
Last date:  2024-12-06
Number of ticker labels: 150
First 5 labels:
   <a target="_blank"><span style="color: blue">AAPL</span></a>
   <a target="_blank"><span style="color: blue">AMGN</span></a>
   <a target="_blank"><span style="color: blue">AMZN</span></a>
   <a target="_blank"><span style="color: blue">AXP</span></a>
   <a target="_blank"><span style="color: blue">BA</span></a>


### Date range and tickers

The data starts on 1962-01-02 and ends on 2024-12-06.

The column `ticker` has **150 different labels**. That is too many. Many labels contain HTML text, such as `<a ...><span ...>AAPL</span></a>`. The same stock appears under several labels. I clean the labels in Section 1.2 before I do anything else.

## 1.2 Detect Data Quality Issues

### Issue 1: Dirty ticker labels

I remove HTML tags, remove extra spaces, and change the text to upper case.
I do this **before** I group rows or look for duplicates. If I did it after, the same stock under two labels would look like two stocks.

I keep the original label in the column `ticker_raw`. I also save a table that maps each original label to the clean ticker. This is my audit record.

In [36]:
df = raw.copy()
df["ticker_raw"] = df["ticker"]
df["ticker"] = (
    df["ticker"]
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
    .str.upper()
)

label_map = (
    df[["ticker_raw", "ticker"]]
    .drop_duplicates()
    .sort_values(["ticker", "ticker_raw"])
    .reset_index(drop=True)
)

print("Labels before:", raw["ticker"].nunique())
print("Tickers after:", df["ticker"].nunique())
print(sorted(df["ticker"].unique()))
label_map.head(10)

Labels before: 150
Tickers after: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,ticker_raw,ticker
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
1,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
2,"<a target=""_blank""><span style=""font-weight: b...",AAPL
3,"<a target=""_blank""><span style=""font-weight: b...",AAPL
4,AAPL,AAPL
5,"<a target=""_blank""><span style=""color: blue"">A...",AMGN
6,"<a target=""_blank""><span style=""color: red"">AM...",AMGN
7,"<a target=""_blank""><span style=""font-weight: b...",AMGN
8,"<a target=""_blank""><span style=""font-weight: b...",AMGN
9,AMGN,AMGN


**Result:** 150 labels became 30 tickers. The table `label_map` shows which original labels map to each clean ticker.

### Issue 2: Duplicate rows

I look for two kinds of duplicates.
1. **Exact duplicates.** Every value is the same. I remove these.
2. **Conflicting keys.** The same `(ticker, ts)` pair appears with different values. I do not remove these without a reason. I investigate them first.

In [37]:
compare_cols = [c for c in df.columns if c != "ticker_raw"]

n_rows = len(df)
n_exact = df.duplicated(subset=compare_cols).sum()
print(f"Rows before: {n_rows:,}")
print(f"Exact duplicate rows: {n_exact:,}")

df = df.drop_duplicates(subset=compare_cols, keep="first").reset_index(drop=True)
print(f"Rows after removing exact duplicates: {len(df):,}")

n_conflict = df.duplicated(subset=["ticker", "ts"], keep=False).sum()
print(f"Rows with a conflicting (ticker, ts) key: {n_conflict:,}")

Rows before: 397,711
Exact duplicate rows: 29,460
Rows after removing exact duplicates: 368,251
Rows with a conflicting (ticker, ts) key: 0


**Result:** I removed about 29,000 exact duplicate rows. After that, no `(ticker, ts)` key appears twice with different values. So there are no conflicts to resolve.

This also tells me the duplicates were copies of the same record. They were not two different prices for one day. Removing them loses no information.

### Setup for the checks

The checks below run on the data **after** the ticker labels are cleaned and exact duplicates are removed. They show the problems that are still in the data **before** any repair. I keep this diagnosis.

Every check is a function. Each function returns a table of issues with the same five columns:
`ticker | issue_type | issue_date | description | severity`.
Then I join all the tables into one Data Quality Report.

In [38]:
# Sort first. Returns and gap checks need each ticker in date order.
df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)

ISSUE_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]
PRICE_COLS = ["open", "high", "low", "close"]
ALL_PRICE_COLS = PRICE_COLS + ["adj_close"]


def make_issues(rows, issue_type, description, severity):
    """Build an issue table (the 5 report columns) from the flagged rows."""
    out = pd.DataFrame({
        "ticker": rows["ticker"].to_numpy(),
        "issue_type": issue_type,
        "issue_date": rows["ts"].to_numpy(),
        "description": description,
        "severity": severity,
    })
    return out[ISSUE_COLS]

### Check 1: Missing values

I count the missing cells in each column. A missing **price** (open, high, low, close) is `high` severity, because I cannot calculate returns without it. A missing `adj_close` or `volume` is `medium` severity. The raw price is still there, so I can often repair the value.

In [39]:
def detect_missing_values(data):
    """Flag missing cells in every price column and in volume."""
    parts = []
    for col in ALL_PRICE_COLS + ["volume"]:
        bad = data[data[col].isna()]
        if bad.empty:
            continue
        severity = "high" if col in PRICE_COLS else "medium"
        parts.append(make_issues(bad, f"missing_{col}", f"{col} is missing", severity))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=ISSUE_COLS)


print(df.isna().sum())
missing_issues = detect_missing_values(df)
missing_issues["issue_type"].value_counts()

ts                0
open              0
high              0
low               0
close             0
adj_close     18374
volume            0
ticker            0
ticker_raw        0
dtype: int64


issue_type
missing_adj_close    18374
Name: count, dtype: int64

**Result:** Only `adj_close` has missing values. The other columns have none. The count is in the table above.

### Check 2: Missing trading days

A missing **cell** and a missing **row** are different problems. Here I look for dates that are absent from a ticker's series.

**Reference calendar.** I do not know the exchange calendar, and weekdays alone include holidays. The homework allows this: use the dates of the supplied `ohlc` table, for each ticker, as the reference calendar. For each ticker, I look only between that ticker's first and last date. A date counts as missing if the `ohlc` table has it and `ohlc_hw` does not.

I only **detect** missing dates. I do not invent prices for them.

In [40]:
def detect_missing_dates(data, calendar):
    """Find dates in the reference calendar that are absent from each ticker's data.

    Returns (issue table, list of summary sentences).
    """
    parts, summary = [], []
    for ticker, cal in calendar.groupby("ticker"):
        have = set(data.loc[data["ticker"] == ticker, "ts"])
        if not have:
            continue
        start, end = min(have), max(have)
        expected = set(cal.loc[cal["ts"].between(start, end), "ts"])
        missing = sorted(expected - have)
        if missing:
            summary.append(
                f"Ticker {ticker} is missing {len(missing)} trading days between {start} and {end}"
            )
            rows = pd.DataFrame({"ticker": ticker, "ts": missing})
            parts.append(make_issues(rows, "missing_date",
                                     "trading date absent from the ticker's series", "medium"))
    issues = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=ISSUE_COLS)
    return issues, summary

**Test on a tiny example.** I remove one date (2024-01-04) from a 5-day series. The detector must find it.

In [41]:
tiny_calendar = pd.DataFrame({
    "ticker": "TEST",
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"],
})
tiny_data = tiny_calendar[tiny_calendar["ts"] != "2024-01-04"]

tiny_issues, tiny_summary = detect_missing_dates(tiny_data, tiny_calendar)
print(tiny_summary)
tiny_issues

['Ticker TEST is missing 1 trading days between 2024-01-02 and 2024-01-08']


,ticker,issue_type,issue_date,description,severity
0,TEST,missing_date,2024-01-04,trading date absent from the ticker's series,medium


**Now the real data.** The reference calendar comes from the table `ohlc` in the same database.

In [42]:
calendar = pd.read_sql_query("SELECT ticker, ts FROM ohlc", conn)
calendar["ticker"] = calendar["ticker"].str.strip().str.upper()
calendar = calendar.drop_duplicates()

gap_issues, gap_summary = detect_missing_dates(df, calendar)
print("Tickers with missing dates:", len(gap_summary))
for line in gap_summary:
    print(line)

Tickers with missing dates: 0


**Result:** No ticker has missing trading dates. The detector works (see the tiny example), so this is a real zero, not a broken check. I do not invent gaps to match the checklist.

### Check 3: Return outliers

1. For each ticker, in date order, I calculate the simple daily return: `(close - prev_close) / prev_close`.
2. For each ticker, I find the mean and the sample standard deviation (`ddof=1`).
3. I flag a day when `abs(return - mean) > 3 * std`. This flags both tails: big gains and big losses.

Rules:
- A return that cannot be calculated (the first day of each ticker) is not flagged.
- If a ticker has a standard deviation of zero or undefined, I flag nothing for it.
- This is a **review flag**. It is not a deletion rule. A flagged day can be a real market move, a stock split, or a data error. I decide in Section 1.3.

**Severity.** I use `medium` when the move is 20% or more in one day. These are the days most likely to be errors. All other flags are `low`.

In [43]:
def add_simple_returns(data):
    """Add column `ret`: simple close-to-close return, calculated inside each ticker."""
    d = data.sort_values(["ticker", "ts"]).copy()
    prev_close = d.groupby("ticker")["close"].shift(1)
    d["ret"] = (d["close"] - prev_close) / prev_close
    return d


def detect_return_outliers(data, n_std=3.0):
    """Flag days where |return - ticker mean| > n_std * ticker std (ddof=1)."""
    d = add_simple_returns(data)
    grp = d.groupby("ticker")["ret"]
    mean = grp.transform("mean")
    std = grp.transform(lambda s: s.std(ddof=1))

    usable = d["ret"].notna() & np.isfinite(d["ret"]) & std.notna() & (std > 0)
    flag = usable & ((d["ret"] - mean).abs() > n_std * std)

    bad = d[flag]
    text = [
        f"close-to-close return {r:+.2%} (ticker mean {m:+.2%}, std {s:.2%})"
        for r, m, s in zip(bad["ret"], mean[flag], std[flag])
    ]
    severity = np.where(bad["ret"].abs() >= 0.20, "medium", "low")
    return make_issues(bad, "return_outlier", text, severity)


outlier_issues = detect_return_outliers(df)
print(len(outlier_issues), "flagged days in", outlier_issues["ticker"].nunique(), "tickers")
outlier_issues["severity"].value_counts()

5056 flagged days in 30 tickers


severity
low       4951
medium     105
Name: count, dtype: int64

**Result:** The screen flags about 5,000 days. All 30 tickers have flags. This is a large number for two reasons. Daily returns have fat tails, so a 3-standard-deviation rule flags many normal days. The screen also covers 60 years of history.

I do not treat all of these as errors. The `medium` rows are the ones to study first.

### Check 4: Impossible price relationships

In a real trading day, the high is the largest price and the low is the smallest price. So these must hold:
- `low <= open <= high`
- `low <= close <= high`

I check five rules. A break in any rule is `high` severity, because the row cannot be true.

| Rule name | Means |
|---|---|
| `high_below_close` | high < close |
| `low_above_close` | low > close |
| `open_above_high` | open > high |
| `open_below_low` | open < low |
| `high_below_low` | high < low |

In [44]:
def detect_price_relationships(data):
    """Flag rows where open, high, low, close break low <= open/close <= high."""
    rules = {
        "high_below_close": data["high"] < data["close"],
        "low_above_close": data["low"] > data["close"],
        "open_above_high": data["open"] > data["high"],
        "open_below_low": data["open"] < data["low"],
        "high_below_low": data["high"] < data["low"],
    }
    parts = []
    for name, mask in rules.items():
        bad = data[mask.fillna(False)]
        if not bad.empty:
            parts.append(make_issues(bad, name, "impossible OHLC relationship", "high"))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=ISSUE_COLS)


price_rel_issues = detect_price_relationships(df)
price_rel_issues["issue_type"].value_counts()

issue_type
open_below_low    5
Name: count, dtype: int64

### Check 5: Zero, negative, or non-finite prices

A stock price must be above zero and must be a real number. I flag any open, high, low, close, or adjusted close that is zero or below, or that is infinite. This is `high` severity.

The raw high/low rules from Check 4 do **not** apply to `adj_close`. The adjusted close can be much lower than the raw prices in old years. So I only test it for zero, negative, and infinite values.

In [45]:
def detect_bad_prices(data):
    """Flag prices that are <= 0 or infinite. (Missing values are Check 1.)"""
    parts = []
    for col in ALL_PRICE_COLS:
        values = data[col]
        mask = values.notna() & ((values <= 0) | ~np.isfinite(values))
        bad = data[mask]
        if not bad.empty:
            parts.append(make_issues(bad, f"nonpositive_or_nonfinite_{col}",
                                     [f"{col} = {v}" for v in bad[col]], "high"))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=ISSUE_COLS)


bad_price_issues = detect_bad_prices(df)
bad_price_issues

,ticker,issue_type,issue_date,description,severity
0,CVX,nonpositive_or_nonfinite_open,1962-01-02,open = 0.0,high
1,JNJ,nonpositive_or_nonfinite_open,1962-01-02,open = 0.0,high
2,MCD,nonpositive_or_nonfinite_open,1966-07-05,open = 0.0,high
3,MRK,nonpositive_or_nonfinite_open,1962-01-02,open = 0.0,high
4,PG,nonpositive_or_nonfinite_open,1962-01-02,open = 0.0,high


### Check 6: Volume problems

- **Zero volume** can mean stale data, but it can also be real on a quiet day in old data. I flag it as `low` severity.
- **Negative volume** is impossible. I flag it as `high` severity.

In [46]:
def detect_volume_issues(data):
    """Flag zero volume (low severity) and negative volume (high severity)."""
    parts = []
    zero = data[data["volume"] == 0]
    if not zero.empty:
        parts.append(make_issues(zero, "zero_volume",
                                 "volume is 0 (possible stale data)", "low"))
    neg = data[data["volume"] < 0]
    if not neg.empty:
        parts.append(make_issues(neg, "negative_volume", "volume is negative", "high"))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=ISSUE_COLS)


volume_issues = detect_volume_issues(df)
volume_issues["issue_type"].value_counts()

issue_type
zero_volume    75
Name: count, dtype: int64

### Test: injected errors

My database has no row where `high < close`, no `low > close`, no negative volume, and no infinite price. A check that finds zero can be correct, or it can be broken. To show that these detectors work, I build a tiny table with **known** errors. Each detector must find its error.

| Date | Injected error |
|---|---|
| 2024-01-02 | `high` (11) below `close` (12) |
| 2024-01-03 | `low` (9) above `close` (8); zero volume; missing `adj_close` |
| 2024-01-04 | `open` is 0; negative volume |
| 2024-01-05 | `open` (10) above `high` (9); `adj_close` is infinite |

In [47]:
demo = pd.DataFrame({
    "ticker": ["X"] * 4,
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
    "open": [10, 10, 0, 10],
    "high": [11, 11, 11, 9],
    "low": [9, 9, 9, 8],
    "close": [12, 8, 10, 9.5],
    "adj_close": [10, np.nan, 10, np.inf],
    "volume": [100, 0, -5, 100],
})

for check in (detect_price_relationships, detect_bad_prices,
              detect_volume_issues, detect_missing_values):
    print(check.__name__)
    print(check(demo)[["issue_type", "issue_date"]].to_string(index=False))
    print()

detect_price_relationships
      issue_type issue_date
high_below_close 2024-01-02
high_below_close 2024-01-05
 low_above_close 2024-01-03
 open_above_high 2024-01-05
  open_below_low 2024-01-04

detect_bad_prices
                        issue_type issue_date
     nonpositive_or_nonfinite_open 2024-01-04
nonpositive_or_nonfinite_adj_close 2024-01-05

detect_volume_issues
     issue_type issue_date
    zero_volume 2024-01-03
negative_volume 2024-01-04

detect_missing_values
       issue_type issue_date
missing_adj_close 2024-01-03



**Result:** Every injected error is found. So the zero counts in the real data (no `high < close`, no `low > close`, no missing dates) are real zeros.

### Data Quality Report

I join all issue tables into one DataFrame. One row is one issue. The same observation can have several issue rows. For example, a row with `open = 0` has both a bad price issue and a price relationship issue.

In [48]:
report = pd.concat(
    [missing_issues, gap_issues, outlier_issues,
     price_rel_issues, bad_price_issues, volume_issues],
    ignore_index=True,
)
report = report.sort_values(["ticker", "issue_date", "issue_type"]).reset_index(drop=True)

print("Total issue rows:", len(report))
report.groupby(["issue_type", "severity"]).size().rename("rows").reset_index()

Total issue rows: 23515


,issue_type,severity,rows
0,missing_adj_close,medium,18374
1,nonpositive_or_nonfinite_open,high,5
2,open_below_low,high,5
3,return_outlier,low,4951
4,return_outlier,medium,105
5,zero_volume,low,75


In [49]:
report.head(10)

,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,close-to-close return +9.23% (ticker mean +0.1...,low
1,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
2,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
3,AAPL,missing_adj_close,1981-02-12,adj_close is missing,medium
4,AAPL,return_outlier,1981-04-02,close-to-close return +8.77% (ticker mean +0.1...,low
5,AAPL,missing_adj_close,1981-04-16,adj_close is missing,medium
6,AAPL,missing_adj_close,1981-05-19,adj_close is missing,medium
7,AAPL,missing_adj_close,1981-06-04,adj_close is missing,medium
8,AAPL,missing_adj_close,1981-07-14,adj_close is missing,medium
9,AAPL,missing_adj_close,1981-07-20,adj_close is missing,medium


### Severity choices

| Severity | Used for | Why |
|---|---|---|
| high | Missing open/high/low/close, impossible price relationships, zero or negative prices, negative volume | The row cannot be true, or I cannot calculate returns from it. |
| medium | Missing `adj_close`, missing volume, missing dates, daily moves of 20% or more | The problem is real or probably real, but I can often repair it from other data. |
| low | Zero volume, return outliers under 20% | These are review flags. They can be normal market behavior. |



## 1.3 Clean the Data

### Repair decisions

| Problem | Rows | What I do | Flag column |
|---|---|---|---|
| `adj_close` missing | 18,374 | Rebuild from the reference table (see below) | `fix_adj_close` |
| `adj_close` far too large (spike) | about 10,400 | Rebuild the same way | `fix_adj_close` |
| `open = 0` | 5 | Use the midpoint of that day's high and low | `fix_open` |
| One-day price doubling that reverses next day, with zero volume | 3 (all TRV) | Confirmed data error. Interpolate between the neighbor days | `fix_spike` |
| Zero volume | 75 | Keep. Zero volume can be real. Only flag it | `flag_zero_volume` |
| Return outliers (3 standard deviations) | about 5,000 | Keep. Real market moves. Only flag them | `flag_return_outlier` |
| Missing open/high/low/close, missing volume, missing dates, broken high/low bounds | 0 in this database | The code handles them anyway. I test this on injected examples | `fix_price_fill`, `fix_volume`, `fix_bounds` |

### Rules I follow
- Sort by ticker and date before any repair. Fill only **inside** one ticker. Never borrow a value from another ticker.
- Drop a row only when all four raw prices are missing. I check this **before** I fill anything.
- A leading gap (no earlier price for the ticker) cannot be filled. Those rows go to a quarantine table with a reason.
- After every repair, I check the high/low bounds again.
- Missing volume is unknown, not zero trading. I fill it with 0 and set a flag, as the homework says.

### Source for the adjusted close repair

The homework allows me to use matched `(ticker, ts)` rows from the `ohlc` table to get a trusted adjustment factor. The factor is `ref_adj_close / ref_close`. I rebuild `adj_close = my close x factor`.

I use this source only for rows that need repair. I do not replace the whole table. All other rows keep the vendor value.

First I check that my raw close is the same as the reference close. If it is, the factor is safe to use.

In [50]:
reference = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn)
reference["ticker"] = reference["ticker"].str.strip().str.upper()
reference = reference.drop_duplicates(["ticker", "ts"])

m = df.merge(reference, on=["ticker", "ts"], how="left", suffixes=("", "_ref"))
print("Rows without a reference row:", m["close_ref"].isna().sum())
print("Rows where raw close differs from reference close:",
      (~np.isclose(m["close"], m["close_ref"], rtol=1e-6)).sum())

Rows without a reference row: 0
Rows where raw close differs from reference close: 0


**Result:** Every row has a reference row, and the raw close is the same in both tables. So the reference factor is trusted.

### Extra check: wrong (not missing) adjusted close values

`adj_close` has a second problem that Section 1.2 did not show. About 10,000 values are **present but wrong**: they are 4 to 24 times too large. I found this because the ratio `adj_close / close` should change slowly. It changes only at dividend and split dates.

**Check:** For each ticker, I compare each day's ratio with the median ratio of the 11 days around it. If the ratio is 1.5 times larger or smaller, I flag the day as a spike. This check does not use the reference table. Supporting evidence: an adjusted close above the raw close is unusual in a price series that is adjusted backward.

In [51]:
def adj_close_spike_mask(data, window=11, tol=1.5):
    """True where adj_close/close is far from the median of nearby days (same ticker)."""
    d = data.sort_values(["ticker", "ts"])
    factor = d["adj_close"] / d["close"]
    typical = factor.groupby(d["ticker"]).transform(
        lambda s: s.rolling(window, center=True, min_periods=3).median())
    ratio = factor / typical
    spike = ratio.notna() & ((ratio > tol) | (ratio < 1 / tol))
    return spike.reindex(data.index)


def detect_adj_close_spikes(data):
    spike = adj_close_spike_mask(data)
    bad = data[spike]
    text = [f"adj_close/close = {a / c:.3f}, far from nearby days"
            for a, c in zip(bad["adj_close"], bad["close"])]
    return make_issues(bad, "adj_close_spike", text, "medium")


adj_spike_issues = detect_adj_close_spikes(df)
print(len(adj_spike_issues), "spike rows")
print("adj_close above close:", (df["adj_close"] > df["close"] * 1.0001).sum())

report = pd.concat([report, adj_spike_issues], ignore_index=True)
report = report.sort_values(["ticker", "issue_date", "issue_type"]).reset_index(drop=True)

10418 spike rows
adj_close above close: 10348


**Result:** About 10,400 rows are spikes. I add them to the Data Quality Report with `medium` severity, because I can repair them from the reference table. One flagged row (TRV, 1978-12-20) is a false alarm, because its raw close, not its adjusted close, is the odd value. The repair gives it the same adjusted close, so this does no harm.

### Cleaning functions

`clean_ohlc` does all repairs in this order. The order matters.

1. Drop rows where all four raw prices are missing.
2. Turn zero, negative, and infinite high/low/close values into missing values. Forward-fill them inside each ticker.
3. Repair `open` (missing or zero) with the midpoint of high and low.
4. Quarantine rows that still have a missing price (leading gaps).
5. Fix rows where high is below the open or close, or low is above them. I widen high and low so that they contain the open and close.
6. Fill missing volume with 0 and set a flag. Quarantine negative volume.
7. Rebuild missing or spiked `adj_close` from the reference factor. Quarantine any row I cannot rebuild.
8. Interpolate the confirmed one-day price errors (spike, then return, with zero volume).
9. Flag return outliers. I do not change them.

The function returns the cleaned table and a quarantine table. Each quarantined row has a reason.

In [52]:
FLAG_COLS = ["fix_price_fill", "fix_open", "fix_bounds", "fix_volume",
             "fix_adj_close", "fix_spike", "flag_zero_volume", "flag_return_outlier"]


def flag_bad_adj_close(data):
    """True where adj_close is missing, not positive, not finite, or a spike."""
    d = data["adj_close"]
    missing = d.isna()
    nonpositive = d.notna() & ~(d > 0)
    not_finite = d.notna() & ~np.isfinite(d.fillna(1))
    return missing | nonpositive | not_finite | adj_close_spike_mask(data)


def find_spike_errors(data, jump=0.20, reversal_tol=0.05):
    """Confirmed one-day errors: close moves 20%+, returns to the old level next day,
    and the day has zero volume."""
    d = data.sort_values(["ticker", "ts"])
    g = d.groupby("ticker")["close"]
    prev, nxt = g.shift(1), g.shift(-1)
    big_move = (d["close"] / prev - 1).abs() >= jump
    comes_back = (nxt / prev - 1).abs() <= reversal_tol
    no_volume = d["volume"] == 0
    return (big_move & comes_back & no_volume).reindex(data.index)


def clean_ohlc(data, reference):
    """Clean the OHLC table. Returns (cleaned table with flags, quarantine table)."""
    d = data.sort_values(["ticker", "ts"]).reset_index(drop=True).copy()
    for c in FLAG_COLS:
        d[c] = False
    removed = []

    # 1. drop rows where ALL raw prices are missing (before any filling)
    all_missing = d[PRICE_COLS].isna().all(axis=1)
    removed.append(d[all_missing].assign(reason="all raw prices missing"))
    d = d[~all_missing].reset_index(drop=True)

    # 2. bad high/low/close -> missing, then forward-fill inside each ticker
    for col in ["high", "low", "close"]:
        bad_value = d[col].notna() & ~(d[col] > 0) | ~np.isfinite(d[col].fillna(1))
        d.loc[bad_value, col] = np.nan
    for col in ["high", "low", "close"]:
        was_missing = d[col].isna()
        d[col] = d.groupby("ticker")[col].ffill()
        d.loc[was_missing & d[col].notna(), "fix_price_fill"] = True

    # 3. open missing or not positive -> midpoint of the same day's high and low
    open_bad = d["open"].isna() | ~(d["open"] > 0) | ~np.isfinite(d["open"].fillna(1))
    d.loc[open_bad, "open"] = (d["high"] + d["low"]) / 2
    d.loc[open_bad, "fix_open"] = True

    # 4. leading gaps cannot be filled -> quarantine
    leading = d[PRICE_COLS].isna().any(axis=1)
    removed.append(d[leading].assign(reason="leading gap: no earlier price for this ticker"))
    d = d[~leading].reset_index(drop=True)

    # 5. recheck bounds after filling; widen high/low to contain open and close
    top = d[PRICE_COLS].max(axis=1)
    bottom = d[PRICE_COLS].min(axis=1)
    bounds_bad = (d["high"] < top) | (d["low"] > bottom)
    d.loc[bounds_bad, "high"] = top[bounds_bad]
    d.loc[bounds_bad, "low"] = bottom[bounds_bad]
    d.loc[bounds_bad, "fix_bounds"] = True

    # 6. volume: missing -> 0 with flag; negative -> quarantine
    d["fix_volume"] = d["volume"].isna()
    d["volume"] = d["volume"].fillna(0)
    negative = d["volume"] < 0
    removed.append(d[negative].assign(reason="negative volume"))
    d = d[~negative].reset_index(drop=True)
    d["flag_zero_volume"] = d["volume"] == 0

    # 7. adj_close: rebuild from the reference factor (only if raw close matches)
    ref = reference.rename(columns={"close": "ref_close", "adj_close": "ref_adj_close"})
    d = d.merge(ref[["ticker", "ts", "ref_close", "ref_adj_close"]],
                on=["ticker", "ts"], how="left")
    needs_fix = flag_bad_adj_close(d)
    close_matches = np.isclose(d["close"], d["ref_close"], rtol=1e-6)
    factor = d["ref_adj_close"] / d["ref_close"]
    can_fix = needs_fix & close_matches & (factor > 0) & np.isfinite(factor)
    d.loc[can_fix, "adj_close"] = d.loc[can_fix, "close"] * factor[can_fix]
    d.loc[can_fix, "fix_adj_close"] = True
    unresolved = needs_fix & ~can_fix
    removed.append(d[unresolved].assign(reason="adj_close unresolved"))
    d = d[~unresolved].reset_index(drop=True)

    # 8. confirmed one-day errors -> interpolate between valid neighbors (no extrapolation)
    spike_rows = find_spike_errors(d)
    for col in ["open", "high", "low", "close", "adj_close"]:
        filled = d[col].where(~spike_rows)
        filled = filled.groupby(d["ticker"]).transform(
            lambda s: s.interpolate(limit_area="inside"))
        d.loc[spike_rows, col] = filled[spike_rows]
    d["fix_spike"] = spike_rows

    # 9. flag return outliers (3 std, ddof=1, both tails). No change to the data.
    ret = d["close"] / d.groupby("ticker")["close"].shift(1) - 1
    mean = ret.groupby(d["ticker"]).transform("mean")
    std = ret.groupby(d["ticker"]).transform(lambda s: s.std(ddof=1))
    d["flag_return_outlier"] = ret.notna() & (std > 0) & ((ret - mean).abs() > 3 * std)

    quarantine = pd.concat(removed, ignore_index=True)
    keep = ["ts", "ticker"] + PRICE_COLS + ["adj_close", "volume"] + FLAG_COLS
    return d[keep], quarantine

### Test: injected errors

My database has no missing prices, no missing volume, and no broken high/low bounds. To show that the repair code works, I build two tiny tickers with known errors.

| Ticker | Row | Injected error | Expected result |
|---|---|---|---|
| AAA | 4th row | close missing | Forward-filled from AAA's earlier day (`fix_price_fill`) |
| AAA | 6th row | open, high, low, close all missing | Removed, with a reason |
| AAA | 8th row | volume missing | Volume 0 and `fix_volume` |
| AAA | 9th row | high below close | High and low widened (`fix_bounds`) |
| BBB | 1st row | close missing | Quarantined as a leading gap. No value is borrowed from AAA |
| BBB | 3rd row | open is 0 | Midpoint of high and low (`fix_open`) |

In [53]:
nan = np.nan
dates = [f"2024-01-{i:02d}" for i in range(2, 12)]


def make_series(ticker, base):
    return pd.DataFrame({"ticker": ticker, "ts": dates, "open": base, "high": base + 1,
                         "low": base - 1, "close": base, "adj_close": base * 0.9,
                         "volume": 1000})


a = make_series("AAA", 100.0)
b = make_series("BBB", 50.0)
a.loc[3, "close"] = nan
a.loc[5, PRICE_COLS] = nan
a.loc[7, "volume"] = nan
a.loc[8, "high"] = a.loc[8, "close"] - 2
b.loc[0, "close"] = nan
b.loc[2, "open"] = 0.0

test_data = pd.concat([a, b], ignore_index=True)
test_ref = test_data[["ticker", "ts", "close", "adj_close"]].copy()
test_ref["close"] = test_ref["close"].fillna(100.0)

test_clean, test_removed = clean_ohlc(test_data, test_ref)
print("rows in:", len(test_data), " kept:", len(test_clean), " quarantined/removed:", len(test_removed))
print(test_removed[["ticker", "ts", "reason"]])
cols = ["ts", "open", "high", "low", "close", "volume", "fix_price_fill", "fix_volume", "fix_bounds"]
print(test_clean.loc[(test_clean["ticker"] == "AAA") & test_clean["ts"].isin([dates[3], dates[7], dates[8]]), cols].to_string())
print(test_clean.loc[(test_clean["ticker"] == "BBB") & (test_clean["ts"] == dates[2]),
                     ["open", "high", "low", "close", "fix_open"]].to_string())

rows in: 20  kept: 18  quarantined/removed: 2
  ticker          ts                                         reason
0    AAA  2024-01-07                         all raw prices missing
1    BBB  2024-01-02  leading gap: no earlier price for this ticker
           ts   open   high   low  close  volume  fix_price_fill  fix_volume  fix_bounds
3  2024-01-05  100.0  101.0  99.0  100.0  1000.0            True       False       False
6  2024-01-09  100.0  101.0  99.0  100.0     0.0           False        True       False
7  2024-01-10  100.0  100.0  98.0  100.0  1000.0           False       False        True
    open  high   low  close  fix_open
10  50.0  51.0  49.0   50.0      True


**Result:** All six injected errors are repaired as expected. The all-missing row is removed and the BBB leading gap is quarantined with a reason. Nothing is copied across tickers.

### Run the cleaning on the real data

In [54]:
cleaned, quarantine = clean_ohlc(df, reference)
print("Rows in:", len(df), " retained:", len(cleaned), " quarantined or removed:", len(quarantine))
cleaned[FLAG_COLS].sum()

Rows in: 368251  retained: 368251  quarantined or removed: 0


fix_price_fill             0
fix_open                   5
fix_bounds                 0
fix_volume                 0
fix_adj_close          28792
fix_spike                  3
flag_zero_volume          75
flag_return_outlier     5175
dtype: int64

### Review of extreme returns

Question: is a flagged day a split or other adjustment, a real market move, or a data error?

**Evidence:** The adjusted close is continuous through a stock split, but the raw close is not. So on a split day, the raw return and the adjusted return are very different. On a normal market move day, the two returns agree. I compare them for every flagged day. I call a day a **market move** when the two returns agree within 10%.

In [55]:
c = cleaned.copy()
c["ret"] = c["close"] / c.groupby("ticker")["close"].shift(1) - 1
c["adj_ret"] = c["adj_close"] / c.groupby("ticker")["adj_close"].shift(1) - 1

flagged = c[c["flag_return_outlier"]].copy()
flagged["gap"] = (1 + flagged["ret"]) / (1 + flagged["adj_ret"]) - 1
flagged["verdict"] = np.where(flagged["gap"].abs() <= 0.10,
                              "market move (raw and adjusted agree)", "needs review")

print(flagged["verdict"].value_counts())
print()
print("Largest moves:")
print(flagged.loc[flagged["ret"].abs().sort_values(ascending=False).index[:8],
                  ["ticker", "ts", "ret", "adj_ret", "gap"]].to_string())
print()
print("Needs review:")
print(flagged.loc[flagged["verdict"] == "needs review",
                  ["ticker", "ts", "close", "adj_close", "ret", "adj_ret", "gap"]].to_string())

verdict
market move (raw and adjusted agree)    5173
needs review                               2
Name: count, dtype: int64

Largest moves:
       ticker          ts       ret   adj_ret       gap
213788    MCD  1968-01-02 -0.753620 -0.755881  0.009261
4999     AAPL  2000-09-29 -0.518695 -0.518686 -0.000018
280845   NVDA  2000-03-07  0.424106  0.424226 -0.000085
281954   NVDA  2004-08-06 -0.352314 -0.352358  0.000067
336480    UNH  2008-10-13  0.347550  0.347482  0.000051
22610    AMZN  2001-11-26  0.344714  0.344714  0.000000
331189    UNH  1987-10-20 -0.342118 -0.342110 -0.000012
22454    AMZN  2001-04-09  0.335723  0.335723  0.000000

Needs review:
       ticker          ts  close  adj_close       ret   adj_ret       gap
2486     AAPL  1990-10-19   0.28       0.28  0.100369  0.382102 -0.203843
264448   MSFT  2004-11-15  27.39      19.05 -0.086086  0.018717 -0.102877


**Result:** Almost all flagged days (5,173 of 5,175) are real market moves. The raw and adjusted returns agree. For example, the AAPL drop of 51.9% on 2000-09-29 appears in both. I keep these rows and flag them. I do not call any of them an error without evidence.

Two days need a note.
- **MSFT, 2004-11-15.** The raw close falls 8.6%, but the adjusted close rises 1.9%. The gap is typical of a large cash dividend, because the adjusted close removes the dividend. I believe this is the $3.00 special dividend that Microsoft paid in 2004, but I did not verify that here. I keep the row and flag it.
- **AAPL, 1990-10-19.** The adjusted close equals the raw close on this one day, while the days around it have an adjusted close about 20% below the close. I cannot tell if this is a vendor error. The reference table has the same value, so I have no better source. I keep the row and state my uncertainty.

The three TRV days are the only confirmed errors. The raw price exactly doubles for one day, the volume is 0, and the price returns to the old level the next day. I repaired them by interpolation.

### Validation and before/after statistics

I check the cleaned table for:
- no missing, non-finite, or non-positive values in the price columns
- finite and non-negative volume
- unique `(ticker, ts)` keys
- correct high/low bounds
- ISO dates (`YYYY-MM-DD`)

Then I account for every row.

In [56]:
def snapshot(x):
    prices = x[ALL_PRICE_COLS]
    top = x[PRICE_COLS].max(axis=1)
    bottom = x[PRICE_COLS].min(axis=1)
    return pd.Series({
        "rows": len(x),
        "missing cells (price/volume)": int(x[ALL_PRICE_COLS + ["volume"]].isna().sum().sum()),
        "non-positive or non-finite prices": int((prices.notna() & ~(prices > 0)).sum().sum()),
        "high/low bound violations": int(((x["high"] < top) | (x["low"] > bottom)).sum()),
        "adj_close spikes": int(adj_close_spike_mask(x).sum()),
        "repeated (ticker, ts) keys": int(x.duplicated(["ticker", "ts"]).sum()),
        "negative volume": int((x["volume"] < 0).sum()),
    })


summary = pd.DataFrame({"before (after dedup)": snapshot(df), "after cleaning": snapshot(cleaned)})
print(summary)

assert cleaned[ALL_PRICE_COLS + ["volume"]].notna().all().all()
assert np.isfinite(cleaned[ALL_PRICE_COLS + ["volume"]]).all().all()
assert (cleaned[ALL_PRICE_COLS] > 0).all().all()
assert not cleaned.duplicated(["ticker", "ts"]).any()
assert (cleaned["volume"] >= 0).all()
assert (cleaned["high"] >= cleaned[["open", "low", "close"]].max(axis=1)).all()
assert (cleaned["low"] <= cleaned[["open", "high", "close"]].min(axis=1)).all()
assert cleaned["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
print("all checks passed")

print(f"{len(raw):,} raw rows - {len(raw) - len(df):,} exact duplicates = {len(df):,}; "
      f"quarantined or removed {len(quarantine):,}; retained {len(cleaned):,}")

                                   before (after dedup)  after cleaning
rows                                             368251          368251
missing cells (price/volume)                      18374               0
non-positive or non-finite prices                     5               0
high/low bound violations                             5               0
adj_close spikes                                  10418               0
repeated (ticker, ts) keys                            0               0
negative volume                                       0               0
all checks passed
397,711 raw rows - 29,460 exact duplicates = 368,251; quarantined or removed 0; retained 368,251


**Result:** All checks pass. Missing cells fall from 18,374 to 0. Rows with non-positive prices fall from 5 to 0. Adjusted close spikes fall from about 10,400 to 0. Bound violations fall from 5 to 0.

**Row accounting:** 397,711 raw rows. 29,460 exact duplicates removed. 368,251 rows left. 0 removed or quarantined. 368,251 rows retained.

### Save the cleaned table

I write the cleaned table to `ohlc_hw_cleaned` in the working copy of the database. The repair flags are columns in the table. I also save the quarantine table (it is empty here). If the tables already exist, they are replaced, so this notebook can run again from the top. The raw table `ohlc_hw` is not changed.

In [57]:
raw_rows_before = conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0]

cleaned.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
quarantine.drop(columns=["ticker_raw"], errors="ignore").to_sql(
    "ohlc_hw_quarantine", conn, if_exists="replace", index=False)
conn.commit()

print(pd.read_sql_query(
    "SELECT COUNT(*) AS rows_saved, MIN(ts) AS first_date, MAX(ts) AS last_date FROM ohlc_hw_cleaned", conn))
print("raw table unchanged:", conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0] == raw_rows_before)

   rows_saved  first_date   last_date
0      368251  1962-01-02  2024-12-06
raw table unchanged: True
